# Divye FE — LightGBM

Same freq + OOF TE + interaction features as `s6e2-divye-fe.ipynb`, but with **LightGBM** instead of CatBoost.

Why this might help:
- CatBoost does sophisticated internal ordered TE during training — so explicit TE adds less new info for CatBoost
- LightGBM uses gradient statistics for categoricals (less powerful than CatBoost's TE) — so explicit TE features are more genuinely novel signal for LGBM
- A strong LGBM + divye FE model also gives a diverse OOF for stacking with the CatBoost divye FE

Baseline LGBM (no FE): OOF AUC ~0.9551  
CatBoost divye FE: OOF AUC 0.95566  LB 0.95381

In [1]:
import subprocess
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

X      = train[ALL_FEATURES]
y      = train['heart_disease'].values
X_test = test[ALL_FEATURES]

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train: {X.shape}  Test: {X_test.shape}')
print(f'LightGBM version: {lgb.__version__}')

Train: (630000, 13)  Test: (270000, 13)
LightGBM version: 4.6.0


In [2]:
def compute_freq_features(train_df, test_df, cols):
    combined = pd.concat([train_df[cols], test_df[cols]])
    tr_freq, te_freq = {}, {}
    for col in cols:
        freq_map = combined[col].value_counts(normalize=True)
        tr_freq[f'{col}_freq'] = train_df[col].map(freq_map).fillna(0).values
        te_freq[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0).values
    return tr_freq, te_freq


def compute_oof_te(train_df, test_df, cols, y, skf):
    global_mean = y.mean()
    oof_te  = {f'{col}_te': np.zeros(len(train_df)) for col in cols}
    test_te = {f'{col}_te': np.zeros(len(test_df))  for col in cols}
    for col in cols:
        key, tr_vals, te_vals, fold_test = f'{col}_te', train_df[col].values, test_df[col].values, []
        for tr_idx, val_idx in skf.split(train_df, y):
            te_map = pd.DataFrame({'v': tr_vals[tr_idx], 't': y[tr_idx]}).groupby('v')['t'].mean()
            oof_te[key][val_idx] = pd.Series(tr_vals[val_idx]).map(te_map).fillna(global_mean).values
            fold_test.append(pd.Series(te_vals).map(te_map).fillna(global_mean).values)
        test_te[key] = np.mean(fold_test, axis=0)
    return oof_te, test_te


def make_interaction_features(te_dict, top_cols):
    return {f'{c1}_x_{c2}': te_dict[c1] * te_dict[c2]
            for c1, c2 in combinations(top_cols, 2)}


def build_augmented(base_df, *dicts):
    df = base_df.copy().reset_index(drop=True)
    for d in dicts:
        for name, vals in d.items():
            df[name] = vals
    return df


print('Frequency encoding...')
tr_freq, te_freq = compute_freq_features(X, X_test, ALL_FEATURES)

print('OOF target encoding...')
oof_te, test_te = compute_oof_te(X, X_test, ALL_FEATURES, y, SKF)

corrs = {col: abs(np.corrcoef(vals, y)[0, 1]) for col, vals in oof_te.items()}
top8  = sorted(corrs, key=corrs.get, reverse=True)[:8]
print(f'Top-8: {top8}')

tr_inter = make_interaction_features(oof_te,  top8)
te_inter = make_interaction_features(test_te, top8)

X_aug      = build_augmented(X,      tr_freq, oof_te,  tr_inter)
X_test_aug = build_augmented(X_test, te_freq, test_te, te_inter)

print(f'Augmented: {X_aug.shape}')

Frequency encoding...
OOF target encoding...
Top-8: ['thallium_te', 'chest_pain_type_te', 'max_hr_te', 'number_of_vessels_fluro_te', 'st_depression_te', 'exercise_angina_te', 'slope_of_st_te', 'sex_te']
Augmented: (630000, 67)


In [3]:
LGBM_PARAMS = dict(
    objective        = 'binary',
    metric           = 'auc',
    learning_rate    = 0.05,
    num_leaves       = 63,
    min_child_samples= 20,
    subsample        = 0.8,
    subsample_freq   = 1,
    colsample_bytree = 0.8,
    device           = 'gpu',
    random_state     = 42,
    verbose          = -1,
)

# LightGBM categorical feature indices (original cat features are columns 0-7)
cat_col_indices = [X_aug.columns.get_loc(c) for c in CAT_FEATURES]

oof_preds  = np.zeros(len(y))
test_folds = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X_aug, y)):
    print(f'\n=== Fold {fold + 1}/5 ===')

    X_tr_base  = X.iloc[tr_idx].reset_index(drop=True)
    X_val_base = X.iloc[val_idx].reset_index(drop=True)
    y_tr, y_val = y[tr_idx], y[val_idx]

    # Recompute freq + TE per fold (proper no-leakage)
    global_mean = y_tr.mean()
    fold_tr_freq, fold_te_freq  = compute_freq_features(X_tr_base, X_test,    ALL_FEATURES)
    _,            fold_val_freq = compute_freq_features(X_tr_base, X_val_base, ALL_FEATURES)

    fold_tr_te, fold_val_te, fold_te_te = {}, {}, {}
    for col in ALL_FEATURES:
        key    = f'{col}_te'
        te_map = pd.DataFrame({'v': X_tr_base[col].values, 't': y_tr}).groupby('v')['t'].mean()
        fold_tr_te[key]  = X_tr_base[col].map(te_map).fillna(global_mean).values
        fold_val_te[key] = X_val_base[col].map(te_map).fillna(global_mean).values
        fold_te_te[key]  = X_test[col].map(te_map).fillna(global_mean).values

    fold_tr_inter  = make_interaction_features(fold_tr_te,  top8)
    fold_val_inter = make_interaction_features(fold_val_te, top8)
    fold_te_inter  = make_interaction_features(fold_te_te,  top8)

    X_tr_aug  = build_augmented(X_tr_base,  fold_tr_freq,  fold_tr_te,  fold_tr_inter)
    X_val_aug = build_augmented(X_val_base, fold_val_freq, fold_val_te, fold_val_inter)
    X_te_aug  = build_augmented(X_test,     fold_te_freq,  fold_te_te,  fold_te_inter)

    dtrain = lgb.Dataset(X_tr_aug,  label=y_tr,  categorical_feature=cat_col_indices, free_raw_data=False)
    dval   = lgb.Dataset(X_val_aug, label=y_val, categorical_feature=cat_col_indices, free_raw_data=False)

    cb = lgb.log_evaluation(period=100)
    es = lgb.early_stopping(stopping_rounds=50, verbose=True)

    model = lgb.train(
        LGBM_PARAMS,
        dtrain,
        num_boost_round = 2000,
        valid_sets      = [dval],
        callbacks       = [cb, es],
    )

    oof_preds[val_idx] = model.predict(X_val_aug)
    test_folds[fold]   = model.predict(X_te_aug)
    print(f'Fold {fold+1}  AUC={roc_auc_score(y_val, oof_preds[val_idx]):.5f}  best_iter={model.best_iteration}')

cv_auc     = roc_auc_score(y, oof_preds)
test_preds = test_folds.mean(axis=0)

print(f'\nOOF AUC (lgbm_divye_fe):    {cv_auc:.5f}')
print(f'Baseline lgbm (no FE):      ~0.95510')
print(f'CatBoost divye_fe:           0.95566')


=== Fold 1/5 ===
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.955583
[200]	valid_0's auc: 0.955727
Early stopping, best iteration is:
[178]	valid_0's auc: 0.955751
Fold 1  AUC=0.95575  best_iter=178

=== Fold 2/5 ===
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.954561
[200]	valid_0's auc: 0.954688
Early stopping, best iteration is:
[164]	valid_0's auc: 0.954716
Fold 2  AUC=0.95472  best_iter=164

=== Fold 3/5 ===
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.955315
[200]	valid_0's auc: 0.955473
Early stopping, best iteration is:
[205]	valid_0's auc: 0.955478
Fold 3  AUC=0.95548  best_iter=205

=== Fold 4/5 ===
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.954917
[200]	valid_0's auc: 0.955094
Early stopping, best iteration is:
[171]	valid_0's auc: 0.95511
Fold 4  AUC=0.95511  best_iter=171

=== Fold 5/5 ===
Training until validation s

In [4]:
np.save('submissions/oof_lgbm_divye_fe.npy', oof_preds)

label = 'lgbm_divye_fe'
fname = f'submissions/{label}.csv'
sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)

desc = f'{label} | cv_auc={cv_auc:.4f}'
print(f'Saved: {fname}')
print(f'desc:  {desc}')

Saved: submissions/lgbm_divye_fe.csv
desc:  lgbm_divye_fe | cv_auc=0.9554


In [5]:
import subprocess
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')

lgbm_divye_fe: submitted
